# Volumetric Method & Lubricate-and-Bleed Case Study
## Influx Migration with Pipe Off Bottom, Boyle's Law Expansion, and Shoe Protection Tiers

### Executive Summary
When a well takes a gas kick with the drill string pulled off bottom, bit plugged, or pipe out of the hole, conventional circulation is impossible. The crew must employ the **Volumetric Method** to allow gas to migrate up the wellbore under controlled conditions until it reaches surface, followed by **Lubricate and Bleed** to replace gas with kill mud.

This case study demonstrates:
1. **Boyle's Law Isothermal Gas Migration:** Tracking gas expansion ($P_1 V_1 = P_2 V_2$) as the bubble rises.
2. **Annular Capacity Transitions:** Why bleed volume per cycle (IADC Rule #35) must adapt to the narrowest annular clearance traversed.
3. **Casing Shoe Pressure Tiers:** Evaluating risk to the shoe (Tier 0 Standard through Tier 4 Supervisor Intervention).
4. **Transition to Lubricate and Bleed:** Converting barrels of kill mud pumped into surface pressure to bleed off.

In [ ]:
from killsheet.volumetric import volume_to_bleed_per_cycle, mud_gradient
from killsheet.volumetric_model import VolumetricModel
from killsheet.lubricate_and_bleed import lubricate_and_bleed_to_volume

# Well Conditions: Vertical Well with Bit 3,000 ft Off Bottom
omw_ppg = 11.0
kmw_ppg = 12.0
td_tvd_ft = 12000
shoe_tvd_ft = 7000
lot_emw_ppg = 15.5
sicp_initial_psi = 600
pit_gain_bbl = 15
working_pressure_psi = 100
safety_margin_psi = 50

### 1. Mud Gradient & Bleed Volume per Cycle
Per IADC WellSharp, during each cycle the casing pressure is allowed to build by a working margin (typically 50-100 psi) as gas migrates, after which mud is bled to restore hydrostatic pressure.

In [ ]:
grad = mud_gradient(omw_ppg)
annular_capacity = 0.0489    # bbl/ft inside 9-5/8" casing

bleed_bbl = volume_to_bleed_per_cycle(working_pressure_psi, annular_capacity, grad)
print(f"Mud Gradient:                {grad:.4f} psi/ft")
print(f"Bleed Volume per Cycle (Pw): {bleed_bbl:.2f} bbl per {working_pressure_psi} psi build")

### 2. Modeling Gas Rise & Shoe Pressure
As gas migrates upward, casing pressure increases to compensate for lost annular hydrostatic head. The casing shoe experiences the highest pressure when the gas bubble top reaches the casing shoe depth.

In [ ]:
lot_psi = (lot_emw_ppg - omw_ppg) * 0.052 * shoe_tvd_ft
shoe_frac_p = 0.052 * lot_emw_ppg * shoe_tvd_ft
print(f"Shoe Formation Breakdown Pressure: {shoe_frac_p:,.0f} psi")
print(f"Allowable Surface MAASP at Shoe:    {lot_psi:,.0f} psi")

### 3. Transition to Lubricate and Bleed
Once the gas bubble reaches the surface BOP, no mud remains above the influx. The crew switches to **Lubricate and Bleed**: pumping kill mud in small batches, waiting for it to settle through gravity segregation, and bleeding off gas pressure equal to the hydrostatic of the lubricated mud.

In [ ]:
# Lubricating 2.0 bbl of 12.0 ppg kill mud into the casing annulus
batch_bbl = 2.0
kmw_grad = mud_gradient(kmw_ppg)
psi_to_bleed = (batch_bbl / annular_capacity) * kmw_grad

print(f"Batch volume pumped:      {batch_bbl:.1f} bbl")
print(f"Hydrostatic column added: {batch_bbl / annular_capacity:.1f} ft")
print(f"Gas pressure to bleed:    {psi_to_bleed:,.0f} psi")